In [1]:
import json
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

path_to_embeddings = "/Users/nad/mobiraph/data/n13_repbase_processed/20_symbols.npz"

path_to_meta = "/Users/nad/mobiraph/data/n13_repbase_processed/metadata_03.json"
path_to_train = "/Users/nad/mobiraph/data/n13_repbase_processed/id_train.txt"
path_to_test = "/Users/nad/mobiraph/data/n13_repbase_processed/id_test.txt"


data = np.load(path_to_embeddings, allow_pickle=True)

names = data["names"]
embeddings = data["embeddings"].reshape(len(data["embeddings"]), -1)

name_to_embedding = dict(zip(names, embeddings))


name_to_type = {}

with open(path_to_meta, "r", encoding="utf-8") as f:
    meta = json.load(f)

# for class_name, class_dict in meta.items():
#     for seq_name in class_dict["sequences"]:
#         name_to_type[seq_name] = class_name


del_classes = {
    "Academ",
    "DNA transposon_other",
    "Kolobok",
    "Troyka",
    "Non-LTR Retrotransposon_other",
    "piggyBac",
}

for name in name_to_embedding:
    if name not in meta:
        continue
    if "superfamily" not in meta[name]:
        continue
    if meta[name]["superfamily"] in del_classes:
        continue

    name_to_type[name] = meta[name]["superfamily"]


with open(path_to_train, "r", encoding="utf-8") as f:
    names_train = [line.strip() for line in f]

with open(path_to_test, "r", encoding="utf-8") as f:
    names_test = [line.strip() for line in f]


train_filtered = [
    name for name in names_train
    if name in name_to_embedding and name in name_to_type
]

test_filtered = [
    name for name in names_test
    if name in name_to_embedding and name in name_to_type
]


X_train = np.array([name_to_embedding[name] for name in train_filtered])
y_train = np.array([name_to_type[name] for name in train_filtered])

X_test = np.array([name_to_embedding[name] for name in test_filtered])
y_test = np.array([name_to_type[name] for name in test_filtered])


print("shape X_train", X_train.shape, "shape y_train", y_train.shape)
print("shape X_test", X_test.shape, "shape y_test", y_test.shape)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


model = LogisticRegression(
    max_iter=2000,
    C=1.0,
    solver="lbfgs",
    penalty="l2",
    class_weight="balanced",
    verbose=1,
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-macro:", f1_score(y_test, y_pred, average="macro"))
print("F1-weighted:", f1_score(y_test, y_pred, average="weighted"))

Accuracy: 0.681467632624445
F1-macro: 0.5409731519564647
F1-weighted: 0.7031657886355638


In [13]:
print("classification report:", classification_report(y_test, y_pred))

classification report:               precision    recall  f1-score   support

         BEL       0.51      0.66      0.57      2346
         CR1       0.21      0.34      0.26       204
       Copia       0.60      0.64      0.62      2664
        DIRS       0.59      0.64      0.62       733
 EnSpm/CACTA       0.70      0.89      0.78       505
       Gypsy       0.87      0.64      0.74      7713
   Harbinger       0.84      0.88      0.86      1044
    Helitron       0.77      0.75      0.76       668
          L1       0.86      0.54      0.66      1275
 Mariner/Tc1       0.62      0.72      0.67       592
        MuDR       0.84      0.85      0.85      1147
         RTE       0.05      0.56      0.10        43
        RTEX       0.56      0.42      0.48       604
        SINE       0.06      0.52      0.11        54
        Tad1       0.01      0.50      0.01         2
         Tx1       0.13      0.32      0.19       177
         hAT       0.93      0.92      0.92      1624

   

In [23]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

param_dist = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [None, 10, 20, 30, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "class_weight": ["balanced", "balanced_subsample"],
}

base_model = RandomForestClassifier(
    n_jobs=-1,
    random_state=42,
)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=30,
    scoring="f1_weighted",
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1,
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV F1 weighted:", search.best_score_)

model = search.best_estimator_

y_pred = model.predict(X_test)

print("F1 macro:", f1_score(y_test, y_pred, average="macro"))
print("F1 weighted:", f1_score(y_test, y_pred, average="weighted"))

Best params: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30, 'class_weight': 'balanced'}
Best CV F1 weighted: 0.8429161341926941
F1 macro: 0.6297729890653768
F1 weighted: 0.8178534520726023


In [2]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

param_dist = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3, 1],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 1.5, 2],
}

base_model = XGBClassifier(
    objective="multi:softmax",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=30,
    scoring="f1_macro",
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1,
)

search.fit(X_train, y_train_enc)

print("Best params:", search.best_params_)
print("Best CV F1 macro:", search.best_score_)

model = search.best_estimator_

y_pred_enc = model.predict(X_test)
y_pred = le.inverse_transform(y_pred_enc)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-macro:", f1_score(y_test, y_pred, average="macro"))

Best params: {'subsample': 1.0, 'reg_lambda': 2, 'reg_alpha': 0.01, 'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.03, 'gamma': 0.3, 'colsample_bytree': 0.6}
Best CV F1 macro: 0.6932470654141542
Accuracy: 0.8315026875438186
F1-macro: 0.6484112771371087


In [3]:
print("F1-macro:", f1_score(y_test, y_pred, average="macro"))
print("F1-weighted:", f1_score(y_test, y_pred, average="weighted"))

F1-macro: 0.6484112771371087
F1-weighted: 0.8252135899180907


In [ ]:
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer
)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.fit_transform(y_test)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        max_iter=1000,
        random_state=42,
        early_stopping=True
    ))
])

param_grid = {
    "mlp__hidden_layer_sizes": [
        (64,),
        (128,),
        (64, 32),
        (128, 64),
        (128, 64, 32)
    ],
    "mlp__activation": ["relu", "tanh"],
    "mlp__alpha": [0.0001, 0.001, 0.01],
    "mlp__learning_rate_init": [0.001, 0.0005, 0.0001],
    "mlp__batch_size": [32, 64, 128]
}

scorer = make_scorer(f1_score, average="macro")

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scorer,
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train_encoded)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest CV Macro-F1:")
print(grid_search.best_score_)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("\nTest Macro-F1:")
print(f1_score(y_test, y_pred, average="macro"))

print("\nTest Weighted-F1:")
print(f1_score(y_test, y_pred, average="weighted"))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

In [5]:
print("Best parameters:")
print(grid_search.best_params_)


Best parameters:
{'mlp__activation': 'relu', 'mlp__alpha': 0.01, 'mlp__batch_size': 32, 'mlp__hidden_layer_sizes': (128, 64), 'mlp__learning_rate_init': 0.0001}


In [7]:
y_pred = le.inverse_transform(y_pred)

In [8]:
y_pred

array(['Copia', 'Copia', 'hAT', ..., 'Copia', 'Gypsy', 'Gypsy'],
      dtype='<U11')

In [9]:
print("\nBest CV Macro-F1:")
print(grid_search.best_score_)

best_model = grid_search.best_estimator_
print("\nTest Macro-F1:")
print(f1_score(y_test, y_pred, average="macro"))

print("\nTest Weighted-F1:")
print(f1_score(y_test, y_pred, average="weighted"))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))


Best CV Macro-F1:
0.6465710166172751

Test Macro-F1:
0.5618094475462428

Test Weighted-F1:
0.7695492098032751

Classification report:
              precision    recall  f1-score   support

         BEL       0.70      0.62      0.66      2346
         CR1       0.25      0.31      0.28       204
       Copia       0.70      0.70      0.70      2664
        DIRS       0.84      0.59      0.70       733
 EnSpm/CACTA       0.88      0.83      0.85       505
       Gypsy       0.82      0.88      0.85      7713
   Harbinger       0.92      0.85      0.88      1044
    Helitron       0.87      0.74      0.80       668
          L1       0.79      0.79      0.79      1275
 Mariner/Tc1       0.64      0.81      0.72       592
        MuDR       0.90      0.81      0.85      1147
         RTE       0.07      0.33      0.12        43
        RTEX       0.79      0.17      0.28       604
        SINE       0.07      0.43      0.12        54
        Tad1       0.01      0.50      0.02         2
